# Day 17 · EDA 综合项目
## 今日学习目标
- 掌握 EDA 标准流程
- 掌握数据清洗方法
- 掌握可视化探索与报告撰写

## 引入
思考：`pd.read_csv` 的 `na_values` 参数有什么用？

在实际项目中，数据里的缺失值常常不以 `NaN` 出现，
而是用 `?`、`N/A`、`无`、`-` 等占位。如果不告诉
pandas 这些符号也算缺失，它们会被当正常字符串读入，
导致后续 `isnull().sum()` 统计不到真实缺失。

今天我们就用一套完整的 EDA 流程来系统掌握数据
探索与清洗的方方面面。

## 第一讲：EDA 流程与数据清洗

### 1.1 EDA 标准流程

EDA（探索性数据分析）一般按以下顺序推进：

1. **数据概览**：`head / shape / info / describe`
2. **数据质量**：缺失值、异常值、重复值
3. **数据修正**：类型转换、填充、删除
4. **可视化探索**：单变量 → 多变量 → 相关性
5. **结论报告**：汇总关键发现

In [ ]:
# 用 print 演示 EDA 流程结构
steps = [
    "1. 数据概览（head/shape/info/describe）",
    "2. 数据质量（缺失值/异常值/重复值）",
    "3. 数据修正（类型转换/填充/删除）",
    "4. 可视化探索（单变量/多变量/相关性）",
    "5. 结论报告（汇总关键发现）",
]
print("EDA 标准流程五大步骤：")
for s in steps:
    print(s)

EDA 标准流程五大步骤：
1. 数据概览（head/shape/info/describe）
2. 数据质量（缺失值/异常值/重复值）
3. 数据修正（类型转换/填充/删除）
4. 可视化探索（单变量/多变量/相关性）
5. 结论报告（汇总关键发现）


### 1.2 数据清洗实战

以下基于 Titanic 数据集演示核心清洗操作。

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("train.csv")

# 清洗前：缺失值统计
print("清洗前缺失值统计：")
print(df[["Survived","Pclass","Sex","Age",
          "SibSp","Parch","Fare","Cabin",
          "Embarked"]].isnull().sum())

# 用中位数填充 Age
df["Age"] = df["Age"].fillna(df["Age"].median())
print(f"\nAge 中位数填充后，缺失值数："
      f"{df['Age'].isnull().sum()}")

# 删除缺失严重的 Cabin 列
df = df.drop(columns=["Cabin"])
print("Cabin 列已删除，当前列：")
print(df.columns.tolist())

# 用众数填充 Embarked
df["Embarked"] = df["Embarked"].fillna(
    df["Embarked"].mode()[0]
)
print("Embarked 用众数填充，剩余缺失值："
      f"{df['Embarked'].isnull().sum()}")

print("\n清洗后缺失值统计：")
print(df.isnull().sum())

清洗前缺失值统计：
Survived      0
Pclass        0
Sex           0
Age         177
SibSp         0
Parch         0
Fare          0
Cabin       687
Embarked      2
dtype: int64

Age 中位数填充后，缺失值数：0
Cabin 列已删除，当前列：
['Survived' 'Pclass' 'Sex' 'Age' 'SibSp' 'Parch' 'Fare' 'Embarked']
Embarked 用众数填充，剩余缺失值：0

清洗后缺失值统计：
Survived    0
Pclass      0
Sex         0
Age         0
SibSp       0
Parch       0
Fare        0
Embarked    0
dtype: int64


> 🔴 易错点：`inplace=True` 与赋值混用
>
> - `df.dropna(inplace=True)` **直接修改原数据**，
>   返回 `None`，不能写 `x = df.dropna(inplace=True)`。
> - `df = df.dropna()` **返回新对象**，必须重新赋值。
> - 建议统一用赋值写法，避免忘记是否需要重绑。

### ✏️ 练习 1

In [ ]:
# 练习：加载 Titanic 数据，完成概览 + 缺失值统计 + 填充
# 要求：
#   1) 读入 train.csv，查看 shape、info、describe
#   2) 对 Age 用中位数填充，Embarked 用众数填充
#   3) 删除 Cabin 列（缺失超 70%）

# ============ 学员代码区 ============
pass
# ===================================

In [ ]:
# 参考答案
df = pd.read_csv("train.csv")
print(df.shape)
df.info()
df = df.drop(columns=["Cabin"])
df["Age"] = df["Age"].fillna(df["Age"].median())
df["Embarked"] = df["Embarked"].fillna(
    df["Embarked"].mode()[0]
)
print("剩余缺失值：")
print(df.isnull().sum().loc[lambda x: x > 0])

In [ ]:
# 练习：检测 Age 列的异常值（箱线图 + 3σ 原则），处理异常值
# 要求：
#   1) 画出 Age 的箱线图（seaborn.boxplot）
#   2) 用 3σ 原则找出 Age 异常值的数量
#   3) 用截断（clip）将异常值拉回边界

# ============ 学员代码区 ============
pass
# ===================================

In [ ]:
# 参考答案
import matplotlib.pyplot as plt
import seaborn as sns

sns.boxplot(x=df["Age"])
plt.title("Age 箱线图")
# plt.show()
print("📊 [此处显示图表]")

mu = df["Age"].mean()
sigma = df["Age"].std()
lower = mu - 3 * sigma
upper = mu + 3 * sigma
outliers = df[(df["Age"] < lower) | (df["Age"] > upper)]
print(f"3σ 异常值数量：{len(outliers)}")

# 截断处理
df["Age"] = df["Age"].clip(lower, upper)
print("截断后 Age 范围："
      f"{df['Age'].min():.1f} ~ {df['Age'].max():.1f}")

## 第二讲：可视化探索与洞察

### 2.1 单变量分布

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Age 直方图
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
df["Age"].hist(bins=30, edgecolor="black")
plt.title("Age 分布")
plt.xlabel("Age")

# Pclass 计数柱状图
plt.subplot(1, 2, 2)
sns.countplot(x="Pclass", data=df)
plt.title("各舱位乘客数")
# plt.show()
print("📊 [此处显示图表]")

📊 [此处显示图表]


### 2.2 多变量关系

In [ ]:
# 按舱位分组求生存率
surv_by_class = df.groupby("Pclass")["Survived"].mean()
print("按舱位分组的生存率：")
print(surv_by_class)
print()

# 柱状图
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
surv_by_class.plot(kind="bar", color="steelblue")
plt.title("各舱位生存率")
plt.ylabel("Survived")
plt.xticks(rotation=0)

# 相关性热力图
plt.subplot(1, 2, 2)
num_cols = df.select_dtypes(include="number")
sns.heatmap(num_cols.corr(), annot=True,
            cmap="coolwarm", fmt=".2f")
plt.title("数值特征相关性")
# plt.show()
print("📊 [此处显示图表]")

按舱位分组的生存率：
Pclass
1    0.629630
2    0.472826
3    0.242363
Name: Survived, dtype: float64

📊 [此处显示图表]


### ✏️ 练习 2

In [ ]:
# 练习：画"性别 × 舱位"分组生存率柱状图，写出 2 条洞察
# 要求：
#   1) 用 groupby(["Sex", "Pclass"]) 计算生存率
#   2) 绘制分组柱状图（hue = Sex）
#   3) 在注释里写出至少 2 条数据洞察

# ============ 学员代码区 ============
pass
# ===================================
# 我的洞察：
# 1. ...
# 2. ...

In [ ]:
# 参考答案
ax = sns.barplot(
    x="Pclass", y="Survived", hue="Sex", data=df
)
plt.title("性别 × 舱位 生存率")
# plt.show()
print("📊 [此处显示图表]")

# 洞察：
# 1. 女性生存率整体远高于男性，符合"女士优先"
# 2. 头等舱女性生存率接近 100%，三等舱男性最低

In [ ]:
# 练习：画 Age - Fare 散点图，按生存状态着色
# 要求：
#   1) 用 seaborn.scatterplot
#   2) hue = "Survived"
#   3) 观察是否存在价格或年龄的分群特征

# ============ 学员代码区 ============
pass
# ===================================

In [ ]:
# 参考答案
sns.scatterplot(
    x="Age", y="Fare", hue="Survived", data=df,
    alpha=0.6
)
plt.title("Age - Fare 散点图（按生存着色）")
# plt.show()
print("📊 [此处显示图表]")

## 第三讲：项目报告撰写

### 3.1 EDA 报告结构

一份合格的 EDA 报告通常包含以下五个部分：

| 章节 | 内容 |
|------|------|
| 数据概览 | 样本量、字段含义、数据类型 |
| 数据质量 | 缺失值、异常值、重复值详情 |
| 关键发现 | 单变量分布 + 多变量关系结论 |
| 可视化 | 至少 3 张有洞察的图 |
| 结论 | 业务含义、建模建议 |

### 3.2 报告撰写要点

- **每张图必须配文字结论**，不能只放图
- 先说"数据告诉了我们什么"，再说业务含义
- 缺失值、异常值的处理方式要在报告里说明
- 最后给出**下一步建议**：如特征工程方向

### ✏️ 综合项目（90 分钟）

### 🎯 项目要求

自选数据集（Iris / Titanic / Housing），
完成一份完整的 EDA 报告并提交 Notebook。

### ✅ Checklist

- [ ] 加载数据并输出 shape/info/describe
- [ ] 统计缺失值并说明处理策略
- [ ] 检测数值列异常值并处理
- [ ] 至少 3 张可视化图并配结论
- [ ] 完成相关性热力图
- [ ] 撰写 5 段式报告（见上）
- [ ] 代码每行 ≤ 100 字符，图表 `plt.show()` 注释

In [ ]:
# 综合项目：自选数据集（Iris/Titanic），完成完整 EDA 报告
# 要求：按 Checklist 六项逐一完成，
#       在 Markdown cell 或注释中撰写报告文本。

# ============ 学员代码区 ============
pass
# ===================================

In [ ]:
# 参考答案骨架（以 Iris 为例）
from sklearn.datasets import load_iris

iris = load_iris(as_frame=True)
df = iris.frame

# 1. 数据概览
print("shape:", df.shape)
print(df.describe())

# 2. 缺失值
print("缺失值：")
print(df.isnull().sum())

# 3. 单变量可视化
df.hist(bins=20, figsize=(8, 6))
plt.suptitle("Iris 各特征分布")
# plt.show()
print("📊 [此处显示图表]")

# 4. 多变量
sns.heatmap(df.corr(), annot=True, cmap="coolwarm")
plt.title("相关性")
# plt.show()
print("📊 [此处显示图表]")

# 5. 结论（写在你的 Markdown cell 中）

## 总结

- **EDA 五步**：概览 → 质量 → 修正 → 可视化 → 报告
- **缺失值**：`isnull().sum()` + `fillna` / `dropna`
- **异常值**：箱线图 + 3σ + `clip`
- **可视化**：hist / countplot / boxplot / scatter /
  barplot / heatmap
- **报告**：数据概览 / 质量 / 发现 / 可视化 / 结论

下一课：我们将进入**数据建模**的旅程！

## ⭐ 挑战题

In [ ]:
# 挑战题：对 Housing 数据集做完整 EDA，
#         找出影响房价的关键因素
# 提示：
#   - 可用 sklearn.datasets.fetch_california_housing
#   - 画两两散点图（pairplot）看线性趋势
#   - 找出与 MedHouseVal 相关系数最大的 3 个特征
#   - 撰写 5 段式报告

# ============ 学员代码区 ============
pass
# ===================================

In [ ]:
# 参考答案提示
from sklearn.datasets import fetch_california_housing

housing = fetch_california_housing(as_frame=True)
df = housing.frame

print("shape:", df.shape)
print(df.describe())

# 与目标列的相关系数排序
target_corr = df.corr()["MedHouseVal"].abs()
target_corr = target_corr.drop("MedHouseVal")
top3 = target_corr.sort_values(ascending=False).head(3)
print("影响房价 Top3 特征：")
print(top3)

sns.heatmap(df.corr(), cmap="coolwarm", center=0)
plt.title("Housing 相关性")
# plt.show()
print("📊 [此处显示图表]")